# Dataset Overview (JSONL)

This notebook loads a JSONL dataset that follows the schema in `configs/schema.json` and reports:
- General statistics (record count, field availability)
- Label distribution
- Text length characteristics (from `word_count`)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Update this path if needed
jsonl_path = Path("../../data/processed/dataset_normalizedlabel.jsonl")

if not jsonl_path.exists():
    candidates = list(Path("data").rglob("*.jsonl"))
    raise FileNotFoundError(
        f"JSONL not found: {jsonl_path}. Update jsonl_path. Candidates: {candidates[:10]}"
    )

df_raw = pd.read_json(jsonl_path, lines=True)


def get_nested(item, *keys):
    cur = item
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return None
        cur = cur[key]
    return cur


# Extract schema fields
rows = []
for record in df_raw.to_dict(orient="records"):
    rows.append(
        {
            "title": get_nested(record, "content", "title"),
            "body_cleaned": get_nested(record, "content", "body_cleaned"),
            "target_label": get_nested(record, "labeling", "target_label"),
            "word_count": get_nested(record, "content", "word_count"),
        }
    )

df = pd.DataFrame(rows)

# General statistics
print(f"Total records: {len(df)}")
print("Non-null field counts:")
print(df[["title", "body_cleaned", "target_label", "word_count"]].notna().sum())

Total records: 111145
Non-null field counts:
title           111145
body_cleaned    111145
target_label    111145
word_count      111145
dtype: int64


In [4]:
# Label distribution
label_counts = df["target_label"].fillna("<missing>").value_counts()
print(label_counts)

# Optional: percentage distribution
label_percent = (label_counts / len(df) * 100).round(2)
print("\nLabel percentages (%):")
print(label_percent)

target_label
kinh_te      18981
giai_tri     18438
the_thao     18127
cong_nghe    17738
du_lich      13673
chinh_tri    12314
giao_duc     11310
suc_khoe       564
Name: count, dtype: int64

Label percentages (%):
target_label
kinh_te      17.08
giai_tri     16.59
the_thao     16.31
cong_nghe    15.96
du_lich      12.30
chinh_tri    11.08
giao_duc     10.18
suc_khoe      0.51
Name: count, dtype: float64


In [ ]:
# Plot label distribution
plt.figure(figsize=(8, 4))
label_counts.sort_values(ascending=False).plot(kind="bar", color="#4C78A8")
plt.title("Label Distribution")
plt.xlabel("Label")
plt.ylabel("Number of articles")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [5]:
# Text length characteristics (word_count)
word_count = pd.to_numeric(df["word_count"], errors="coerce")

print(f"Average word_count: {word_count.mean():.2f}")
print(f"Min word_count: {word_count.min()}")
print(f"Max word_count: {word_count.max()}")

# Optional: quick distribution summary
print("\nword_count summary:")
print(word_count.describe())

Average word_count: 684.61
Min word_count: 1
Max word_count: 13571

word_count summary:
count    111145.000000
mean        684.610851
std         426.709584
min           1.000000
25%         430.000000
50%         592.000000
75%         825.000000
max       13571.000000
Name: word_count, dtype: float64


In [ ]:
# Plot word_count histogram
plt.figure(figsize=(8, 4))
word_count.dropna().plot(kind="hist", bins=40, color="#F58518", edgecolor="black")
plt.title("Word Count Distribution")
plt.xlabel("Word count")
plt.ylabel("Number of articles")
plt.tight_layout()
plt.show()